# FASE 4 - Deep Learning (Modelo Secuencial GRU)

## Paso 1: Carga de Datos y Scores Híbridos (ML)
Cargamos el dataset limpio para construir las secuencias históricas y el CSV con los **scores híbridos (70% Colaborativo + 30% Contenido)** exportados desde la fase de Machine Learning (`ml_scores_hibrido.csv`).

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# 1. Cargar el dataset limpio para las secuencias
ruta_dataset_limpio = Path('../data/2020-Apr-L.csv')
df_dl = pd.read_csv(ruta_dataset_limpio, parse_dates=['event_time'])

# 2. Cargar el CSV con el resultado híbrido de la fase ML
ruta_ml_hibrido = Path('../data/ml_scores_hibrido.csv')
df_scores_hibrido = pd.read_csv(ruta_ml_hibrido)

# 3. Mapeo de IDs a índices numéricos para los Embeddings (0 a N-1)
usuarios_unicos = df_dl['user_id'].unique()
productos_unicos = df_dl['product_id'].unique()

user_to_index = {user_id: idx for idx, user_id in enumerate(usuarios_unicos)}
product_to_index = {product_id: idx for idx, product_id in enumerate(productos_unicos)}

df_dl['user_idx'] = df_dl['user_id'].map(user_to_index)
df_dl['product_idx'] = df_dl['product_id'].map(product_to_index)

df_scores_hibrido['user_idx'] = df_scores_hibrido['user_id'].map(user_to_index)
df_scores_hibrido['product_idx'] = df_scores_hibrido['product_id'].map(product_to_index)

# 4. Definición de variables de entorno reales
NUM_USUARIOS = len(usuarios_unicos)
NUM_PRODUCTOS = len(productos_unicos)
LONGITUD_SECUENCIA = 10
EMBEDDING_DIM = 64

print("---- UNIFICACIÓN COMPLETADA ----")
print(f"Usuarios (Dimensión User Embedding): {NUM_USUARIOS}")
print(f"Productos (Dimensión Product Embedding): {NUM_PRODUCTOS}")

---- UNIFICACIÓN COMPLETADA ----
Usuarios (Dimensión User Embedding): 1379296
Productos (Dimensión Product Embedding): 60525


## Paso 2: Generación de los Tensores de Entrenamiento (Uso del Score Híbrido)
Aquí es donde se **cruza la secuencia del usuario con los resultados híbridos de ML**. 
Por cada usuario, tomamos sus N últimos productos como contexto temporal. Luego evaluamos los candidatos recomendados por la capa ML, usando el `score_ml_refinado` como entrada adicional para el modelo.

In [2]:
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split

# Agrupar los historiales de los usuarios ordenados por tiempo
historial_usuarios = df_dl.sort_values('event_time').groupby('user_idx')['product_idx'].apply(list).to_dict()

X_secuencias = []
X_usuarios = []
X_candidatos = []
X_ml_scores = []
y_targets = []

# Iteramos sobre las recomendaciones híbridas pre-calculadas en ML
# El CSV ahora incluye la columna 'target' generada en ml.ipynb
for index, row in df_scores_hibrido.dropna(subset=['user_idx', 'product_idx']).iterrows():
    uid = int(row['user_idx'])
    candidato_id = int(row['product_idx'])
    ml_score = row['score_ml_refinado']
    target = int(row['target'])
    
    # Obtenemos el historial del usuario y eliminamos el candidato para evitar leakage
    historial = historial_usuarios.get(uid, [])
    historial_filtrado = [p for p in historial if p != candidato_id]
    
    X_usuarios.append(uid)
    X_candidatos.append(candidato_id)
    X_ml_scores.append(ml_score)
    X_secuencias.append(historial_filtrado[-LONGITUD_SECUENCIA:])
    y_targets.append(target)

# Convertimos a tensores numpy listos para Keras
X_secuencias = pad_sequences(X_secuencias, maxlen=LONGITUD_SECUENCIA, padding='pre')
X_usuarios = np.array(X_usuarios)
X_candidatos = np.array(X_candidatos)
X_ml_scores = np.array(X_ml_scores)
y_targets = np.array(y_targets)

print(f"Tensores generados: {len(X_usuarios)} muestras listas.")
print(f"Distribución de clases: {np.bincount(y_targets)}")

# ---------- DIVISIÓN TRAIN/VAL POR USUARIO ----------
# Evita leakage: usuarios de validación no se ven durante entrenamiento.
usuarios_unicos_idx = np.unique(X_usuarios)
train_users, val_users = train_test_split(
    usuarios_unicos_idx,
    test_size=0.2,
    random_state=42
)
train_mask = np.isin(X_usuarios, train_users)
val_mask = np.isin(X_usuarios, val_users)

X_secuencias_train = X_secuencias[train_mask]
X_usuarios_train = X_usuarios[train_mask]
X_candidatos_train = X_candidatos[train_mask]
X_ml_scores_train = X_ml_scores[train_mask]
y_train = y_targets[train_mask]

X_secuencias_val = X_secuencias[val_mask]
X_usuarios_val = X_usuarios[val_mask]
X_candidatos_val = X_candidatos[val_mask]
X_ml_scores_val = X_ml_scores[val_mask]
y_val = y_targets[val_mask]

print(f"Train: {len(y_train)} muestras, Usuarios: {len(train_users)}")
print(f"Val:   {len(y_val)} muestras, Usuarios: {len(val_users)}")


Tensores generados: 53031 muestras listas.
Distribución de clases: [50000  3031]
Train: 42411 muestras, Usuarios: 800
Val:   10620 muestras, Usuarios: 200


## Paso 3: Definición de Entradas del Modelo
Ahora sí, con los datos preparados (tensores `X`), definimos las puertas de entrada (Inputs) de la arquitectura Keras.

In [3]:
from tensorflow.keras.layers import Input

input_secuencia = Input(shape=(LONGITUD_SECUENCIA,), name='secuencia_productos')
input_usuario = Input(shape=(1,), name='usuario_id')
input_candidato = Input(shape=(1,), name='producto_candidato')
input_ml_score = Input(shape=(1,), name='ml_score')

## Paso 4: Capas de Embedding y Red GRU
Transformamos los índices en vectores densos y pasamos la secuencia temporal por la red recurrente GRU.

In [4]:
from tensorflow.keras.layers import Embedding, GRU, Flatten

emb_producto_layer = Embedding(input_dim=NUM_PRODUCTOS, output_dim=EMBEDDING_DIM, name="emb_producto")
emb_usuario_layer = Embedding(input_dim=NUM_USUARIOS, output_dim=EMBEDDING_DIM, name="emb_usuario")

emb_secuencia = emb_producto_layer(input_secuencia)
emb_usuario = Flatten()(emb_usuario_layer(input_usuario))
emb_candidato = Flatten()(emb_producto_layer(input_candidato))

# Capa GRU que captura el comportamiento secuencial (Vector DL)
vector_dl = GRU(units=64, name="Capa_GRU")(emb_secuencia)

## Paso 5: Fusión de Contextos (Predicción Final)
Concatenamos el Vector DL (comportamiento), el perfil del usuario, el producto candidato y la afinidad calculada en ML.

In [5]:
from tensorflow.keras.layers import Concatenate, Dense, Dropout

caracteristicas_fusionadas = Concatenate(name="fusion_caracteristicas")([
    vector_dl,       # Información secuencial
    emb_usuario,     # Rasgos generales del usuario
    emb_candidato,   # Rasgos del producto
    input_ml_score   # Peso proveniente del Score Híbrido
])

x = Dense(128, activation='relu')(caracteristicas_fusionadas)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
output_score = Dense(1, activation='sigmoid', name="score_final")(x)

## Paso 6: Compilación y Entrenamiento del Modelo
Ensamblamos el modelo y lo entrenamos usando los tensores de datos que preparamos en el Paso 2.

In [6]:
from tensorflow.keras.models import Model

modelo_dl = Model(
    inputs=[input_secuencia, input_usuario, input_candidato, input_ml_score],
    outputs=output_score,
    name="UrbanSoul_Hybrid_GRU"
)

modelo_dl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

modelo_dl.summary()

# ENTRENAMIENTO DEL MODELO
# Usamos train/val divididos por usuario para evitar data leakage.
historial = modelo_dl.fit(
    x=[
        X_secuencias_train,
        X_usuarios_train,
        X_candidatos_train,
        X_ml_scores_train
    ],
    y=y_train,
    validation_data=([
        X_secuencias_val,
        X_usuarios_val,
        X_candidatos_val,
        X_ml_scores_val
    ], y_val),
    epochs=5,
    batch_size=256
)

Model: "UrbanSoul_Hybrid_GRU"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ secuencia_productos │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ usuario_id          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ producto_candidato  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_producto        │ (None, 1, 64)     │  3,873,600 │ secuencia_produc… │
│ (Embedding)         │                   │            │ producto_candida… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_usuario         │ (None, 1, 64)     │ 88,274,944 │ usuario_id[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Capa_GRU (GRU)      │ (None, 64)        │     24,960 │ emb_producto[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 64)        │          0 │ emb_usuario[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 64)        │          0 │ emb_producto[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ml_score            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_caracterist… │ (None, 193)       │          0 │ Capa_GRU[0][0],   │
│ (Concatenate)       │                   │            │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ ml_score[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     24,832 │ fusion_caracteri… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_final (Dense) │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 92,206,657 (351.74 MB)

 Trainable params: 92,206,657 (351.74 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
166/166 ━━━━━━━━━━━━━━━━━━━━ 125s 695ms/step - accuracy: 0.9615 - auc: 0.9370 - loss: 0.1254 - val_accuracy: 0.9841 - val_auc: 0.9970 - val_loss: 0.0407
Epoch 2/5
166/166 ━━━━━━━━━━━━━━━━━━━━ 118s 708ms/step - accuracy: 0.9969 - auc: 0.9995 - loss: 0.0117 - val_accuracy: 0.9854 - val_auc: 0.9965 - val_loss: 0.0353
Epoch 3/5
166/166 ━━━━━━━━━━━━━━━━━━━━ 116s 695ms/step - accuracy: 0.9989 - auc: 0.9999 - loss: 0.0042 - val_accuracy: 0.9767 - val_auc: 0.9946 - val_loss: 0.0508
Epoch 4/5
166/166 ━━━━━━━━━━━━━━━━━━━━ 117s 707ms/step - accuracy: 0.9993 - auc: 1.0000 - loss: 0.0024 - val_accuracy: 0.9801 - val_auc: 0.9920 - val_loss: 0.0687
Epoch 5/5
166/166 ━━━━━━━━━━━━━━━━━━━━ 119s 717ms/step - accuracy: 0.9996 - auc: 1.0000 - loss: 0.0017 - val_accuracy: 0.9768 - val_auc: 0.9931 - val_loss: 0.0547


## Paso 7: Generación de Recomendaciones Finales y Exportación

In [7]:
print("Generando predicciones finales con el modelo GRU entrenado...")

# Predicciones sobre el conjunto de validación (puedes cambiar a todos los datos)
predicciones_finales = modelo_dl.predict([
    X_secuencias_val,
    X_usuarios_val,
    X_candidatos_val,
    X_ml_scores_val
])

# Agregamos las predicciones al dataframe de candidatos de validación
df_resultados_dl = df_scores_hibrido.dropna(subset=['user_idx', 'product_idx']).copy()
df_resultados_dl = df_resultados_dl.iloc[val_mask].copy()
df_resultados_dl = df_resultados_dl.iloc[:len(predicciones_finales)].copy()

# Guardamos el score arrojado por la red neuronal
df_resultados_dl['score_dl_final'] = predicciones_finales

# Ordenamos para obtener el Top-K por usuario
df_top_recomendaciones = (
    df_resultados_dl
    .sort_values(['user_id', 'score_dl_final'], ascending=[True, False])
    .groupby('user_id')
    .head(50)
)

# Agrupamos los IDs de los productos en formato lista para evaluación
df_exportar = df_top_recomendaciones.groupby('user_id')['product_id'].apply(list).reset_index()
df_exportar.rename(columns={'product_id': 'recomendaciones'}, inplace=True)

# Exportamos el CSV que leerá evaluacion.ipynb
ruta_salida = '../data/predicciones_hibrido_final.csv'
df_exportar.to_csv(ruta_salida, index=False)
print(f"¡Exportación exitosa! Resultados guardados en: {ruta_salida}")
print(f"Total de usuarios con recomendaciones: {len(df_exportar)}")

Generando predicciones finales con el modelo GRU entrenado...
332/332 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step
¡Exportación exitosa! Resultados guardados en: ../data/predicciones_hibrido_final.csv
Total de usuarios con recomendaciones: 200
